In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

*Consultă [referința API](https://docs.quantum.ibm.com/api/functions/qunova-chemistry)*

> **Note:** Funcțiile Qiskit sunt o funcționalitate experimentală disponibilă numai utilizatorilor IBM Quantum&reg; Premium Plan, Flex Plan și On-Prem (prin IBM Quantum Platform API) Plan. Acestea se află în stadiu de previzualizare și pot suferi modificări.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.45.0
```
</AccordionItem>
</Accordion>

## Prezentare generală
În chimia cuantică, problema structurii electronice se concentrează pe găsirea soluțiilor ecuației Schrödinger electronice — funcțiile de undă cuantice care descriu comportamentul electronilor sistemului. Aceste funcții de undă sunt vectori de amplitudini complexe, fiecare amplitudine corespunzând contribuției unei configurații electronice posibile.

Starea fundamentală este funcția de undă cu cea mai mică energie a sistemului și are o importanță deosebită în studiul sistemelor moleculare. Abordarea cea mai precisă pentru calcularea stării fundamentale ia în considerare toate configurațiile electronice posibile, însă aceasta devine imposibil de gestionat pentru sisteme mai mari, deoarece numărul configurațiilor crește exponențial cu dimensiunea sistemului.

Handover Iterative Variational Quantum Eigensolver (HI-VQE) este o metodă hibridă cuantic-clasică inovatoare pentru estimarea precisă a stării fundamentale a sistemelor moleculare. Integrează hardware cuantic cu calcul clasic, utilizând procesoare cuantice pentru a explora eficient configurațiile electronice candidate și calculând funcția de undă rezultantă pe calculatoare clasice. Prin generarea de funcții de undă compacte, dar precise din punct de vedere chimic, HI-VQE îmbunătățește cercetarea și descoperirile în chimia cuantică și știința materialelor.

![Imagine care prezintă o privire de ansamblu asupra algoritmului HI-VQE al Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE reduce complexitatea computațională a problemei structurii electronice prin estimarea eficientă a stării fundamentale cu o precizie ridicată. Se concentrează pe un subset atent selectat al celor mai relevante configurații electronice, optimizând atât precizia, cât și eficiența.

Combinând punctele forte ale calculatoarelor clasice și cuantice, HI-VQE rafinează și îmbunătățește iterativ estimarea curentă a funcției de undă. Tehnicile sale unice de construire a subspatiului ajută la eficientizarea selecției configurațiilor, astfel încât utilizatorii să aibă un control computational mai mare și o precizie îmbunătățită în simulările de chimie cuantică.

Dacă dorești să înveți mai în profunzime despre algoritm, poți [citi lucrarea de cercetare asociată](https://arxiv.org/abs/2503.06292).

## Descriere
Numărul de configurații electronice pentru un sistem molecular crește exponențial cu dimensiunea sistemului. Cu toate acestea, pentru anumite stări electronice, cum ar fi starea fundamentală, este obișnuit ca doar o mică fracțiune din configurații să contribuie semnificativ la energia stării. Metodele de interacțiune de configurație selectată (SCI) exploatează această sparsitate pentru a reduce costurile computaționale prin identificarea și concentrarea pe cele mai relevante configurații. Acest subset de configurații este denumit subspațiu.

HI-VQE valorifică eficiența inerentă a calculatoarelor cuantice pentru reprezentarea sistemelor moleculare în scopul facilitării căutării în subspațiu. Integrează subrutine clasice și cuantice pentru a rezolva problema structurii electronice cu o precizie ridicată. Spre deosebire de metodele SCI cuantice existente, HI-VQE combină antrenamentul variațional, construirea iterativă a subspațiului și filtrarea configurațiilor prin pre-diagonalizare pentru a îmbunătăți eficiența prin reducerea măsurătorilor cuantice, a iterațiilor și a costurilor de diagonalizare clasică. HI-VQE poate fi astfel aplicat unor sisteme moleculare mai mari care necesită mai mulți Qubiți și reduce costul rezolvării unei probleme de o dimensiune dată la același grad de precizie.

![Imagine care prezintă o descriere detaliată a fiecărui pas din algoritmul HI-VQE al Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

Pentru a calcula starea fundamentală a unui sistem, HI-VQE utilizează mai întâi pachetul clasic de chimie PySCF pentru a genera o reprezentare moleculară din datele de intrare furnizate de utilizator, cum ar fi geometria moleculară și alte informații moleculare. Intră apoi într-o buclă de optimizare hibridă cuantic-clasică, rafinând iterativ un subspațiu pentru a reprezenta optim starea fundamentală, minimizând în același timp numărul de configurații incluse. Bucla continuă până când sunt îndeplinite criterii de convergență, cum ar fi dimensiunea subspațiului sau stabilitatea energiei, după care sunt furnizate funcția de undă a stării fundamentale calculate și energia. Aceste rezultate pot fi utilizate pentru a construi suprafețe de energie potențială precise și pentru a efectua analize suplimentare ale sistemului.

Bucla de optimizare se concentrează pe ajustarea parametrilor unui Circuit cuantic pentru a genera un subspațiu de înaltă calitate. HI-VQE oferă trei opțiuni de circuit cuantic: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) și [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). Optimizarea este inițializată aproape de starea de referință Hartree-Fock datorită adecvabilității sale generale. Circuit-ul este apoi executat pe un dispozitiv cuantic și configurațiile sunt eșantionate din starea cuantică rezultantă înainte de a fi returnate ca șiruri binare. Datorită zgomotului dispozitivului cuantic, unele configurații eșantionate pot fi invalide din punct de vedere fizic, neconservând numărul de electroni sau spinul. HI-VQE abordează aceasta folosind procesul de recuperare a configurațiilor din pachetul [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), astfel încât utilizatorii pot fie să corecteze configurațiile invalide, fie să le elimine.

Configurațiile valide parcurg apoi un pas opțional de filtrare pentru a le elimina pe cele estimate să contribuie minim. Aceasta reduce dimensiunea subspațiului, scăzând astfel costul pasului de diagonalizare. Dacă filtrarea este activată, atunci se construiește un Hamiltonian de subspațiu preliminar din configurațiile valide și se efectuează o diagonalizare cu criterii de terminare foarte relaxate. Deși precizia amplitudinilor rezultante pentru fiecare configurație este scăzută, aceasta este eficientă pentru a prezice ce configurații să fie excluse din subspațiu în această iterație și este rapidă de calculat.

Configurațiile selectate sunt adăugate la subspațiu, iar Hamiltonianul sistemului este proiectat în acest subspațiu. Subspațiul se actualizează iterativ, păstrând cele mai relevante configurații de-a lungul iterațiilor. Această abordare contrastează cu metodele alternative deoarece Circuit-ul cuantic nu trebuie să aproximeze întreaga stare fundamentală la fiecare pas.

Ulterior, Hamiltonianul de subspațiu este diagonalizat clasic pentru a obține cea mai mică valoare proprie și vectorul propriu corespunzător, reprezentând o aproximare a stării fundamentale și a energiei sale. Pe măsură ce calitatea subspațiului se îmbunătățește prin iterații, starea fundamentală calculată aproximează mai bine starea fundamentală reală. Un pas suplimentar de filtrare poate fi efectuat în acest punct pentru a elimina din subspațiu orice configurații care nu contribuie substanțial la starea fundamentală calculată. Acest pas asigură că subspațiul transferat în iterația următoare este cât mai compact posibil. Aceasta este evaluată pe baza amplitudinilor returnate de diagonalizare, deoarece acestea reprezintă contribuția de importanță a fiecărei configurații la starea fundamentală calculată.

O verificare a convergenței determină apoi dacă antrenamentul suplimentar ar îmbunătăți rezultatele. Dacă da, se efectuează un pas opțional de expansiune clasică, parametrii Circuit-ului cuantic sunt actualizați pentru a minimiza în continuare energia calculată și procesul se repetă. Pasul de expansiune clasică generează configurații suplimentare pentru subspațiu, completând configurațiile eșantionate de pe dispozitivul cuantic. Identifică mai întâi configurația cu amplitudinea cea mai mare în rezultatele diagonalizării, înainte de a genera noi configurații cu excitații simple și duble din configurația identificată. Numărul dorit din aceste configurații este apoi adăugat la subspațiu.

Odată ce s-a determinat că iterațiile au convergat, HI-VQE returnează starea fundamentală calculată (sub forma stărilor din subspațiu și a amplitudinilor lor în funcția de undă a stării fundamentale), energia sa și o măsură a varianței energiei care indică dacă starea calculată formează o stare proprie a Hamiltonianului sistemului.

Utilizatorii pot decide Circuit-ul cuantic utilizat și numărul de shot-uri efectuate pentru fiecare circuit cuantic, precum și controla dimensiunea subspațiului sau activa generarea clasică de configurații suplimentare pentru a asista configurațiile generate cuantic. În acest fel, utilizatorii pot adapta comportamentul HI-VQE pentru aplicațiile dorite.

## Licențiere
Te rog să reții că utilizarea acestei Funcții Qiskit este limitată la probleme care necesită cel mult 20 de qubiți, dacă nu se obține o licență care acordă o limită mai mare.

Te rog să trimiți un e-mail la [qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com) dacă dorești să faci o solicitare privind obținerea unei licențe.
## Noțiuni introductive
Mai întâi, [solicită acces la funcție](https://forms.office.com/r/zN3hvMTqJ1).
Apoi, autentifică-te folosind [cheia ta API IBM Quantum&reg;](http://quantum.cloud.ibm.com/) și, presupunând că ți-ai [salvat deja contul](/guides/functions#install-qiskit-functions-catalog-client) în mediul local, selectează Funcția Qiskit după cum urmează:

In [2]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Exemplu

Primul exemplu arată cum se calculează energia stării fundamentale pentru o moleculă de NH3 folosind algoritmul HI-VQE.

#### Definirea geometriei moleculare și a opțiunilor

Geometria moleculară a NH3 este furnizată cu coordonate carteziene separate prin ";" pentru fiecare atom.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Opțiuni suplimentare pot fi definite și furnizate pentru sistemul molecular în formatul de dicționar următor.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Execută funcția cu intrările de geometrie și opțiuni.

In [ ]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits,
    # for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

Este o idee bună să afișezi ID-ul jobului funcției, astfel încât să poată fi furnizat în solicitările de suport dacă ceva merge greșit.

In [6]:
print("Job ID:", job.job_id)

Job ID: e5ced6f2-fd1d-4244-a6aa-bd27cfb0cdee


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


Acest exemplu utilizează apoi 16 qubiți cu 8 orbitali ai bazei sto3g pentru o moleculă de NH3.
Verifică [starea](/guides/functions#check-job-status) sau returnează [rezultatele](/guides/functions#retrieve-results) workload-ului tău Qiskit Function după cum urmează:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824448589364075, 0.009527106392132133, 6.854074372058527e-08, 3.591500190038039e-07, 0.0012975231577544268, 2.310159709002111e-05, ...], 'energy': -55.52108557170985, 'energy_history': [-55.51901898989887, -55.52056881448526, -55.52065046778772, -55.520690696813716, -55.520691108428, -55.520708448092634, ...], 'energy_variance': 3.066239097617371e-10, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [ ]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: "
    f"{abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.06246299427914437 mHa
Sampled Number of States: 1936


După finalizarea jobului, rezultatele pot fi obținute cu instanța `result()`.

In [10]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [11]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

Pentru a accesa energia stării fundamentale, folosește cheia `"energy"`. Cheia `"eigenvector"` furnizează coeficienții CI cu notația bitstring corespunzătoare a configurației electronice stocate cu `"states"` din rezultate.

In [12]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: ["runner.UnknownRuntimeError: 'An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance. -- https://docs.quantum.ibm.com/errors#1500'\n"]

In [13]:
job.status()

'ERROR'

## Performanță

Această secțiune prezintă calculele de referință demonstrate ale HI-VQE pentru un caz cu 24 de qubiți pentru Li2S, un caz cu 40 de qubiți pentru o moleculă N2 și un caz cu 44 de qubiți pentru un sistem FeP-NO.

#### Curba suprafeței de energie potențială de disociere pentru o moleculă Li2S cu 24 de qubiți

Curba PES este prezentată cu referința FCI și estimarea inițială din RHF, împreună cu eroarea de energie față de referința FCI.

![Imagine care arată că HI-VQE produce soluții în limita acurateței chimice față de o curbă PES de referință clasică pentru sistemul Li2S](../docs/images/guides/qunova-chemistry/Li2S_PES.avif).

Calculele au fost efectuate cu următoarele geometrii și opțiuni.